# Liquidity Supply Model — Simplest Specification

**Research question:** Does unpredictable LP entry compress per-capita fee revenue in the HDRN/USDC Uniswap V3 pool?

**Specification:** Two-step structural econometric model (see `notes/structural-econometrcics/specs/2026-03-02-hdrn-usdc-competition-risk.md`).

- **Step 1:** Entry predictability regression — decompose liquidity growth into predictable and unpredictable components
- **Step 2:** Fee-share impact regression — test whether unpredictable entry compresses fee share
- **Step 3:** Economic magnitude — variance decomposition

**Data:** 82,799 swap-level observations from Uniswap V3 subgraph (Feb 2022 — Mar 2026). Liquidity and feeGrowth from PoolHourData (hourly join).

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import statsmodels.api as sm
from statsmodels.regression.linear_model import OLS
from statsmodels.stats.diagnostic import het_breuschpagan, acorr_ljungbox

plt.style.use('classic')
plt.rcParams.update({
    'figure.figsize': (10, 4),
    'font.size': 9,
    'axes.grid': True,
    'grid.alpha': 0.3,
})

In [ ]:
# Load observations (swap-level with hourly liquidity/feeGrowth join)
df = pd.read_csv('../data/hdrn_usdc/observations.csv')
df['datetime'] = pd.to_datetime(df['timestamp'], unit='s', utc=True)
df = df.sort_values('timestamp').reset_index(drop=True)

print(f'Observations: {len(df):,}')
print(f'Period: {df.datetime.min()} to {df.datetime.max()}')
print(f'\nRaw variable summary:')
df[['pi_n', 'l_active_n', 'dlog_l_n', 'volume_n', 'gas_n', 'dp_n']].describe()

## 1. Variable Scaling & Construction

### Scaling decisions

| Variable | Raw units | Scaling | Rationale |
|----------|-----------|---------|----------|
| `pi_n` | feeGrowth delta / Q128 | Normalize to `pi_n / pi_bar` (rolling mean) | Remove level effects, isolate deviations |
| `l_active_n` | Raw liquidity (uint128) | Already differenced as `dlog_l_n` | Log-difference is scale-free |
| `volume_n` | USD | `ln(1 + volume_n)` for log-linear; raw for linear | Right-skewed, log stabilizes variance |
| `gas_n` | Gwei | `ln(gas_n)` for log-linear; Gwei for linear | Heavy right tail (pre-EIP-1559 spikes up to 42K Gwei) |
| `dp_n` | log price change | Used as-is | Already in log-difference form |
| `dlog_l_n` | log liquidity change | Used as-is | Already in log-difference form |

### `pi_bar` window choice

Using a 200-swap rolling mean. This is a sensitivity parameter — we test 100 and 500 in Section 6.

In [ ]:
# --- Variable construction ---

# pi_bar: rolling mean of pi_n (200-swap window)
PI_BAR_WINDOW = 200
df['pi_bar'] = df['pi_n'].rolling(window=PI_BAR_WINDOW, min_periods=PI_BAR_WINDOW).mean()

# Normalized fee share: pi_n / pi_bar
# Guard against division by zero (pi_bar == 0 in early obs or dead periods)
df['pi_norm'] = np.where(df['pi_bar'] > 0, df['pi_n'] / df['pi_bar'], np.nan)

# Log fee share for log-linear spec
df['ln_pi_norm'] = np.where(df['pi_norm'] > 0, np.log(df['pi_norm']), np.nan)

# Lagged liquidity growth for Step 1
df['dlog_l_lag1'] = df['dlog_l_n'].shift(1)

# Log volume (for log-linear Model A)
df['ln_volume'] = np.log1p(df['volume_n'])

# Log gas (for log-linear specs)
df['ln_gas'] = np.log(df['gas_n'])

# Drop rows with NaN from rolling window and lag
df_reg = df.dropna(subset=['pi_norm', 'ln_pi_norm', 'dlog_l_lag1']).copy()
# Drop infinite values
df_reg = df_reg.replace([np.inf, -np.inf], np.nan).dropna(
    subset=['pi_norm', 'ln_pi_norm', 'dlog_l_lag1', 'ln_gas']
)

print(f'Regression sample: {len(df_reg):,} obs (dropped {len(df) - len(df_reg):,} from rolling window/NaN)')
print(f'\nConstructed variable summary:')
df_reg[['pi_norm', 'ln_pi_norm', 'dlog_l_n', 'dlog_l_lag1', 'volume_n', 'ln_volume', 'gas_n', 'ln_gas', 'dp_n']].describe()

## 2. Data Diagnostics

Before running regressions, verify:
1. Fee share variation is not dominated by a few extreme observations
2. Liquidity growth has meaningful variation (not constant)
3. No perfect collinearity among regressors

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(12, 6))

# pi_norm distribution
axes[0, 0].hist(df_reg['pi_norm'].clip(0, 5), bins=100, color='k', alpha=0.7)
axes[0, 0].set_title('pi_n / pi_bar (clipped at 5)')
axes[0, 0].axvline(1.0, color='gray', linestyle='--', linewidth=0.8)

# dlog_l_n distribution
axes[0, 1].hist(df_reg['dlog_l_n'].clip(-5, 5), bins=100, color='k', alpha=0.7)
axes[0, 1].set_title('DlogL_n (clipped at +/-5)')

# volume distribution (log)
axes[0, 2].hist(df_reg['ln_volume'], bins=100, color='k', alpha=0.7)
axes[0, 2].set_title('ln(1 + Volume_n)')

# gas distribution (log)
axes[1, 0].hist(df_reg['ln_gas'], bins=100, color='k', alpha=0.7)
axes[1, 0].set_title('ln(gas_n) [Gwei]')

# dp_n distribution
axes[1, 1].hist(df_reg['dp_n'].clip(-5, 5), bins=100, color='k', alpha=0.7)
axes[1, 1].set_title('DP_n (clipped at +/-5)')

# Time series of pi_norm (subsample for visibility)
sub = df_reg.iloc[::50]  # every 50th obs
axes[1, 2].plot(sub['datetime'], sub['pi_norm'].clip(0, 5), 'k-', linewidth=0.3)
axes[1, 2].set_title('pi_n/pi_bar over time')
axes[1, 2].tick_params(axis='x', rotation=30)

plt.tight_layout()
plt.show()

# Correlation matrix of regressors
corr_vars = ['dlog_l_n', 'dlog_l_lag1', 'dp_n', 'gas_n', 'ln_gas', 'volume_n', 'ln_volume']
print('\nCorrelation matrix (regressors):')
print(df_reg[corr_vars].corr().round(3).to_string())

## 3. Step 1 — Entry Predictability Regression

**Equation:**
$$\Delta\log L_n = \beta_0 + \beta_1 \Delta\log L_{n-1} + \beta_2 \cdot gas_n + \beta_3 \cdot \Delta P_n + u_n$$

**Purpose:** Decompose liquidity growth into predictable (fitted values) and unpredictable ($\hat{u}_n$) components. The residual $\hat{u}_n$ is the competition shock used in Step 2.

**Inference:** HAC (Newey-West) standard errors with automatic bandwidth selection. No distributional assumptions on errors.

In [ ]:
# Step 1: Entry predictability regression
# DlogL_n = beta_0 + beta_1 * DlogL_{n-1} + beta_2 * gas_n + beta_3 * DP_n + u_n

y_step1 = df_reg['dlog_l_n']
X_step1 = sm.add_constant(df_reg[['dlog_l_lag1', 'gas_n', 'dp_n']])

# OLS with Newey-West HAC standard errors
# Bandwidth: Newey-West automatic (maxlags ~ N^(1/3))
nw_lags = int(np.ceil(len(y_step1) ** (1/3)))
print(f'Newey-West bandwidth: {nw_lags} lags')

model_step1 = OLS(y_step1, X_step1).fit(cov_type='HAC', cov_kwds={'maxlags': nw_lags})
print(model_step1.summary())

# Extract residuals = u_hat_n (unpredictable competition shock)
df_reg = df_reg.copy()
df_reg['u_hat'] = model_step1.resid

print(f'\nResidual u_hat summary:')
print(f'  mean: {df_reg["u_hat"].mean():.6f} (should be ~0)')
print(f'  std:  {df_reg["u_hat"].std():.6f}')
print(f'  R²:   {model_step1.rsquared:.4f}')

### Step 1 Diagnostics

Check:
1. Residual autocorrelation (Ljung-Box) — if strong, model may need more lags
2. Residual heteroskedasticity (Breusch-Pagan) — HAC SEs handle this, but good to know
3. Residual distribution — no normality required, but check for extreme outliers

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(12, 3.5))

# Residual histogram
axes[0].hist(df_reg['u_hat'].clip(-5, 5), bins=100, color='k', alpha=0.7)
axes[0].set_title('Step 1 residuals (u_hat)')
axes[0].axvline(0, color='gray', linestyle='--', linewidth=0.8)

# ACF of residuals
sm.graphics.tsa.plot_acf(df_reg['u_hat'].values, lags=30, ax=axes[1],
                         color='k', vlines_kwargs={'color': 'k'})
axes[1].set_title('ACF of u_hat')

# Residual vs fitted
axes[2].scatter(model_step1.fittedvalues, df_reg['u_hat'], s=0.3, c='k', alpha=0.3)
axes[2].axhline(0, color='gray', linestyle='--', linewidth=0.8)
axes[2].set_xlabel('Fitted values')
axes[2].set_ylabel('Residuals')
axes[2].set_title('Residuals vs Fitted')

plt.tight_layout()
plt.show()

# Ljung-Box test for residual autocorrelation
lb = acorr_ljungbox(df_reg['u_hat'], lags=10, return_df=True)
print('Ljung-Box test (H0: no autocorrelation):')
print(lb.to_string())

# Breusch-Pagan test for heteroskedasticity
bp_stat, bp_pval, _, _ = het_breuschpagan(model_step1.resid, model_step1.model.exog)
print(f'\nBreusch-Pagan test: stat={bp_stat:.2f}, p-value={bp_pval:.4f}')
print('(HAC SEs are robust to heteroskedasticity regardless)')

## 4. Step 2 — Fee-Share Impact Regression

Two models estimated:

**Model A** (volume as control):
$$\pi_n / \bar{\pi} = \alpha + \theta \hat{u}_n + \delta_1 \text{Volume}_n + \delta_2 \text{gas}_n + \varepsilon_n$$

**Model B** (perfect volume elasticity — volume drops out):
$$\pi_n / \bar{\pi} = \alpha + \theta \hat{u}_n + \delta_2 \text{gas}_n + \varepsilon_n$$

Both estimated in linear and log-linear form.

**Key coefficient:** $\theta < 0$ means unpredictable LP entry compresses fee share.

**Generated regressor note:** $\hat{u}_n$ is estimated from Step 1. Standard errors in Step 2 should be adjusted (Murphy-Topel or bootstrap). We report HAC SEs as a lower bound and note this caveat.

In [ ]:
# Step 2: Fee-share impact regressions
# All with Newey-West HAC standard errors

results = {}

# --- LINEAR SPECIFICATIONS ---

# Model A (linear): pi_norm = alpha + theta*u_hat + delta1*volume + delta2*gas + eps
y_lin = df_reg['pi_norm']
X_A_lin = sm.add_constant(df_reg[['u_hat', 'volume_n', 'gas_n']])
results['A_linear'] = OLS(y_lin, X_A_lin).fit(cov_type='HAC', cov_kwds={'maxlags': nw_lags})

# Model B (linear): pi_norm = alpha + theta*u_hat + delta2*gas + eps
X_B_lin = sm.add_constant(df_reg[['u_hat', 'gas_n']])
results['B_linear'] = OLS(y_lin, X_B_lin).fit(cov_type='HAC', cov_kwds={'maxlags': nw_lags})

# --- LOG-LINEAR SPECIFICATIONS ---

# Model A (log-linear): ln(pi_norm) = alpha + theta*u_hat + delta1*ln_volume + delta2*ln_gas + eps
y_log = df_reg['ln_pi_norm']
X_A_log = sm.add_constant(df_reg[['u_hat', 'ln_volume', 'ln_gas']])
results['A_loglinear'] = OLS(y_log, X_A_log).fit(cov_type='HAC', cov_kwds={'maxlags': nw_lags})

# Model B (log-linear): ln(pi_norm) = alpha + theta*u_hat + delta2*ln_gas + eps
X_B_log = sm.add_constant(df_reg[['u_hat', 'ln_gas']])
results['B_loglinear'] = OLS(y_log, X_B_log).fit(cov_type='HAC', cov_kwds={'maxlags': nw_lags})

# --- RESULTS TABLE ---
print('Step 2 Results: theta (competition shock coefficient)')
print('=' * 70)
print(f'{"Model":20s} {"theta":>10s} {"SE(HAC)":>10s} {"t-stat":>10s} {"p-value":>10s} {"R²":>8s}')
print('-' * 70)
for name, res in results.items():
    theta = res.params['u_hat']
    se = res.bse['u_hat']
    t = res.tvalues['u_hat']
    p = res.pvalues['u_hat']
    r2 = res.rsquared
    print(f'{name:20s} {theta:>10.6f} {se:>10.6f} {t:>10.3f} {p:>10.4f} {r2:>8.4f}')
print('=' * 70)
print('\nH0: theta >= 0 (no competition effect)')
print('H1: theta < 0 (unpredictable entry compresses fee share)')

In [ ]:
# Full regression tables for each specification
for name, res in results.items():
    print(f'\n{"=" * 70}')
    print(f'Model: {name}')
    print(f'{"=" * 70}')
    print(res.summary())

## 5. Step 3 — Economic Magnitude (Variance Decomposition)

$$\text{Competition risk share} = \frac{\text{Var}(\hat{\theta} \cdot \hat{u}_n)}{\text{Var}(\pi_n / \bar{\pi})}$$

This measures what fraction of fee share variance is attributable to unpredictable competition shocks.

In [ ]:
print('Variance Decomposition: Competition Risk Share')
print('=' * 60)

for name, res in results.items():
    theta = res.params['u_hat']
    
    # Variance of the competition component
    var_competition = np.var(theta * df_reg['u_hat'])
    
    # Variance of the dependent variable
    if 'loglinear' in name:
        var_y = np.var(df_reg['ln_pi_norm'])
    else:
        var_y = np.var(df_reg['pi_norm'])
    
    share = var_competition / var_y if var_y > 0 else 0
    
    print(f'{name:20s}  Var(theta*u_hat)={var_competition:.6g}  Var(y)={var_y:.6g}  Share={share:.4f} ({100*share:.2f}%)')

print('\nInterpretation: share > 5% is economically meaningful for a single risk factor.')

## 6. Specification Tests

| # | Test | Mathematical statement |
|---|------|----------------------|
| 1 | Competition compresses fee share | $\theta < 0$ (one-sided t-test) |
| 2 | Competition risk is economically significant | $\text{Var}(\theta \hat{u}) / \text{Var}(y) > 0.05$ |
| 3 | Volume adds no information (Model A vs B) | LR test: $\delta_1 = 0$ |

In [ ]:
from scipy import stats

print('Specification Test 1: theta < 0 (one-sided)')
print('=' * 60)
for name, res in results.items():
    t = res.tvalues['u_hat']
    # One-sided p-value (left tail: theta < 0)
    p_one_sided = stats.t.cdf(t, df=res.df_resid)
    verdict = 'REJECT H0' if p_one_sided < 0.05 else 'FAIL TO REJECT'
    print(f'{name:20s}  t={t:>8.3f}  p(one-sided)={p_one_sided:.4f}  [{verdict}]')

print(f'\nSpecification Test 3: Volume adds information? (LR test, Model A vs B)')
print('=' * 60)

# Linear: compare A_linear vs B_linear
for form in ['linear', 'loglinear']:
    llf_a = results[f'A_{form}'].llf
    llf_b = results[f'B_{form}'].llf
    lr_stat = 2 * (llf_a - llf_b)
    lr_pval = 1 - stats.chi2.cdf(lr_stat, df=1)  # 1 restriction
    verdict = 'Volume matters' if lr_pval < 0.05 else 'Volume drops out'
    print(f'{form:15s}  LR stat={lr_stat:>10.3f}  p-value={lr_pval:.4f}  [{verdict}]')

## 7. Sensitivity Analysis

### 7.1 pi_bar window sensitivity
Re-estimate with window = 100 and 500 to check coefficient stability.

In [ ]:
print('Sensitivity: pi_bar window choice')
print('=' * 70)
print(f'{"Window":>8s} {"theta(B_lin)":>14s} {"SE":>10s} {"t":>8s} {"R²":>8s} {"N":>8s}')
print('-' * 70)

for window in [100, 200, 500]:
    _df = df.copy()
    _df['pi_bar_w'] = _df['pi_n'].rolling(window=window, min_periods=window).mean()
    _df['pi_norm_w'] = np.where(_df['pi_bar_w'] > 0, _df['pi_n'] / _df['pi_bar_w'], np.nan)
    _df['dlog_l_lag1'] = _df['dlog_l_n'].shift(1)
    _df = _df.dropna(subset=['pi_norm_w', 'dlog_l_lag1']).replace([np.inf, -np.inf], np.nan).dropna(subset=['pi_norm_w'])
    
    # Step 1
    _y1 = _df['dlog_l_n']
    _X1 = sm.add_constant(_df[['dlog_l_lag1', 'gas_n', 'dp_n']])
    _m1 = OLS(_y1, _X1).fit()
    _df = _df.copy()
    _df['u_hat_w'] = _m1.resid
    
    # Step 2 Model B linear
    _y2 = _df['pi_norm_w']
    _X2 = sm.add_constant(_df[['u_hat_w', 'gas_n']])
    _nw = int(np.ceil(len(_y2) ** (1/3)))
    _m2 = OLS(_y2, _X2).fit(cov_type='HAC', cov_kwds={'maxlags': _nw})
    
    print(f'{window:>8d} {_m2.params["u_hat_w"]:>14.6f} {_m2.bse["u_hat_w"]:>10.6f} {_m2.tvalues["u_hat_w"]:>8.3f} {_m2.rsquared:>8.4f} {len(_df):>8,}')

### 7.2 Sub-period analysis

Test coefficient stability across:
- Pre-Merge (Feb 2022 — Sep 2022): PoW, variable block times
- Post-Merge (Sep 2022 onwards): PoS, fixed 12s blocks

In [ ]:
MERGE_TS = 1663224179  # Sep 15, 2022

print('Sub-period analysis: theta stability')
print('=' * 70)
print(f'{"Period":>15s} {"N":>8s} {"theta(B_lin)":>14s} {"SE(HAC)":>10s} {"t":>8s} {"p":>8s}')
print('-' * 70)

for label, mask in [('Pre-Merge', df_reg['timestamp'] < MERGE_TS),
                     ('Post-Merge', df_reg['timestamp'] >= MERGE_TS),
                     ('Full sample', pd.Series(True, index=df_reg.index))]:
    sub = df_reg[mask].copy()
    if len(sub) < 500:
        print(f'{label:>15s} {len(sub):>8,}  -- too few observations --')
        continue
    
    # Step 1 on sub-period
    _y1 = sub['dlog_l_n']
    _X1 = sm.add_constant(sub[['dlog_l_lag1', 'gas_n', 'dp_n']])
    _m1 = OLS(_y1, _X1).fit()
    sub['u_hat_sub'] = _m1.resid
    
    # Step 2 Model B linear
    _y2 = sub['pi_norm']
    _X2 = sm.add_constant(sub[['u_hat_sub', 'gas_n']])
    _nw = int(np.ceil(len(_y2) ** (1/3)))
    _m2 = OLS(_y2, _X2).fit(cov_type='HAC', cov_kwds={'maxlags': _nw})
    
    theta = _m2.params['u_hat_sub']
    se = _m2.bse['u_hat_sub']
    t = _m2.tvalues['u_hat_sub']
    p = _m2.pvalues['u_hat_sub']
    print(f'{label:>15s} {len(sub):>8,} {theta:>14.6f} {se:>10.6f} {t:>8.3f} {p:>8.4f}')

### 7.3 Reduced-form comparison

Estimate without the two-step decomposition — use raw `DlogL_n` instead of `u_hat_n`. If structural and reduced-form estimates are similar, the decomposition validates.

In [ ]:
# Reduced form: pi_norm = alpha + gamma * DlogL_n + delta * gas_n + eps
y_rf = df_reg['pi_norm']
X_rf = sm.add_constant(df_reg[['dlog_l_n', 'gas_n']])
model_rf = OLS(y_rf, X_rf).fit(cov_type='HAC', cov_kwds={'maxlags': nw_lags})

print('Reduced-form comparison')
print('=' * 60)
print(f'{"Specification":20s} {"Coef(competition)":>18s} {"SE":>10s} {"t":>8s}')
print('-' * 60)
print(f'{"Structural (u_hat)":20s} {results["B_linear"].params["u_hat"]:>18.6f} {results["B_linear"].bse["u_hat"]:>10.6f} {results["B_linear"].tvalues["u_hat"]:>8.3f}')
print(f'{"Reduced-form (DlogL)":20s} {model_rf.params["dlog_l_n"]:>18.6f} {model_rf.bse["dlog_l_n"]:>10.6f} {model_rf.tvalues["dlog_l_n"]:>8.3f}')
print(f'\nIf signs agree and magnitudes are similar, the two-step decomposition is validated.')
print(f'If different, the decomposition into predictable vs unpredictable entry is empirically important.')

## 8. Summary & Interpretation

In [ ]:
# Collect key results
best = results['B_linear']  # Model B linear as baseline
theta_hat = best.params['u_hat']
theta_se = best.bse['u_hat']
theta_t = best.tvalues['u_hat']
theta_p = best.pvalues['u_hat']

var_comp = np.var(theta_hat * df_reg['u_hat'])
var_y = np.var(df_reg['pi_norm'])
comp_share = var_comp / var_y if var_y > 0 else 0

print('Summary of Key Findings')
print('=' * 60)
print(f'Sample: {len(df_reg):,} swap-level observations')
print(f'Pool: HDRN/USDC 1% (Uniswap V3)')
print(f'Period: {df_reg.datetime.min().strftime("%Y-%m-%d")} to {df_reg.datetime.max().strftime("%Y-%m-%d")}')
print()
print(f'Step 1 (Entry predictability): R² = {model_step1.rsquared:.4f}')
print(f'  -> {100*model_step1.rsquared:.1f}% of liquidity growth is predictable from lagged growth, gas, price')
print(f'  -> {100*(1-model_step1.rsquared):.1f}% is unpredictable (competition shock)')
print()
print(f'Step 2 (Fee-share impact, Model B linear):')
print(f'  theta = {theta_hat:.6f} (SE = {theta_se:.6f})')
print(f'  t-stat = {theta_t:.3f}, p-value = {theta_p:.4f}')
sign = 'NEGATIVE (competition compresses fees)' if theta_hat < 0 else 'POSITIVE (unexpected)'
print(f'  Sign: {sign}')
print()
print(f'Step 3 (Economic magnitude):')
print(f'  Competition risk share = {100*comp_share:.2f}% of fee share variance')
print(f'  Interpretation: {"Economically meaningful" if comp_share > 0.05 else "Modest effect"}')